<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</font></center>
<center><font size=6>MLS 3 - Retrieval Augmented Generation</font></center>

<center><p float="center">
  <img src="https://images.pexels.com/photos/7163956/pexels-photo-7163956.jpeg" alt="health-insurance" width="640"/>
</p></center>

<center><font size=6>MeridianHealth</center></font>

# Problem Statement

## Business Context

MeridianHealth Insurance is a mid-sized health insurance carrier headquartered in Hartford, Connecticut, licensed across 22 states. The company serves approximately 2.4 million covered lives across 47 active plan designs (individual, family, employer-sponsored, supplemental).

A direct sales force of roughly 4,800 licensed representatives handles live customer conversations. Customers ask about benefits, waiting periods, deductibles, copays, exclusions, and claim processes. Simple queries are handled from existing FAQ documents. Harder queries require pulling information from multiple policy documents, combining it, and giving a single coherent answer. Today this synthesis is done manually: searching documents during the call, putting the customer on hold, or answering from memory.

As MeridianHealth grows its portfolio and expands into new states, manual multi-document lookup does not scale. The primary concern is not speed. It is that manual lookup is prone to human error, and in a regulated insurance context, answering a policy question incorrectly or partially carries real consequences.

## Risks

Four risks need to be addressed before an AI-assisted retrieval and answering system can be trusted in this environment:

### 1. Faithfulness Risk

Answers must be grounded strictly in the policy documents. If the system introduces a benefit or condition that does not exist in the source material, the representative may pass along a claim that the company cannot honour. In insurance, this is a misrepresentation issue with regulatory and financial consequences.

### 2. Retrieval Completeness Risk

Many policy questions require information from more than one document. If the system retrieves relevant context from the policy brochure but misses a critical exclusion or waiting period buried in the terms and conditions, the representative may provide a partial answer.A partial answer in insurance is often worse than no answer because it creates false confidence.

### 3. Retrieval Precision Risk

The RAG system provides both a generated answer and the retrieved source chunks that support it. The representative uses these chunks to verify the answer before relaying it to the customer.If the retrieved chunks themselves are not relevant, the entire verification mechanism breaks down. The representative either trusts an unverifiable answer or discards the system output entirely.


### 4. Answer Relevancy Risk

The generated response must actually address the question asked by the representative. Returning tangentially related policy language, even if it is technically faithful to the source documents, does not help the representative move the conversation forward.

## Objective


The objective is to build an **evaluated Retrieval-Augmented Generation (RAG) pipeline** that ingests MeridianHealth's policy documentation, retrieves the most relevant context for a representative's query, generates a source-grounded answer, and demonstrates through measurable evaluation metrics that each successive improvement in the pipeline justifies its added complexity.

The pipeline should:

1. **Ingest policy documents** such as policy brochures, terms and conditions, claim exclusion references, and claim process guides into a vector store so that relevant clauses and sections can be retrieved semantically.

2. **Retrieve and generate answers** by accepting a representative's query, retrieving the most relevant policy context from across documents, and generating a grounded answer using a large language model.

3. **Provide source references** by returning the supporting source passages and document references alongside every answer, allowing the representative to verify the response and explain it confidently to the customer.

4. **Handle complex queries** including comparative and cross-document questions, such as differences between plan benefits, waiting period conditions across products, or combined reimbursement and cashless claim guidance.

## Success Criteria

The pipeline routes every generated answer through a scoring stage. Based on the **composite evaluation score**, the answer is routed to one of three actions:

1. **Auto answer (score above 75%).** The generated answer and its supporting source chunks are trustworthy enough to present directly to the representative for immediate use with the customer.

2. **Assisted review (score 50% to 75%).** The answer is partially grounded but not fully reliable. The representative is shown the retrieved source chunks so they can manually review the relevant passages and formulate their own answer with the right information in front of them.

3. **Manual lookup (score below 50%).** The system output is not reliable enough to act on. The representative falls back to the current process: looking up the answer manually or escalating to the back-office helpdesk.

The weights for the composite score are **informed by past evaluation results and the relative importance of each metric to the business objectives**. Faithfulness is given the highest weight because avoiding unsupported or misleading information is the most critical requirement in this use case.

$$
\text{Composite Score} =
0.35 \times \text{Faithfulness}
+ 0.25 \times \text{Contextual Recall}
+ 0.20 \times \text{Contextual Precision}
+ 0.20 \times \text{Answer Relevancy}
$$

| **Business Objective** | **Technical Metric** | **Threshold** | **Rationale** |
|---|---|---:|---|
| Avoid misrepresentation | Faithfulness | **≥ 0.85** | Measures whether generated answers are grounded in the retrieved policy documents and do not introduce unsupported claims. A higher bar than the typical 0.80 is warranted because unfaithful answers in insurance directly translate to misrepresentation risk, claim disputes, and regulatory exposure. |
| Surface all relevant information | Contextual Recall | **≥ 0.80** | Measures whether the retrieval step captures all the relevant pieces of information needed to answer the query. For multi-document questions, missing a waiting period clause or exclusion from a second document can result in a partial answer. |
| Ensure retrieved chunks are verifiable | Contextual Precision | **≥ 0.75** | Measures whether the retrieved chunks are actually relevant to the query. Since representatives use these chunks to verify the generated answer, low precision makes the verification mechanism unreliable. |
| Answer the question that was asked | Answer Relevancy | **≥ 0.75** | Measures whether the generated response addresses the specific question rather than returning tangentially related policy language. A response that is faithful but off-topic does not help the representative move the conversation forward. |
| Reduce manual lookup burden | % of queries auto answered | **≥ 50%** | Below 50%, most queries still require the representative to manually review chunks or fall back to the helpdesk, which limits the operational value of the pipeline. Above 50%, the majority of queries are answered reliably enough to use directly, while the remaining queries can be handled through assisted review or manual lookup. |

## Dataset Description

**Policy Documents**

| Document | Pages | Description |
| -| -| -|
| `MH_Plan_Benefits_Coverage_Guide_2026.pdf` | 5 | Summarizes Plan Year 2026 cost-sharing parameters across ACA metal tiers (Bronze, Silver, Gold, Platinum), Essential Health Benefits, prescription drug formulary tiers, and Supplemental Shield cash indemnity benefits[cite: 4] |
| `MH_Claim_Exclusion_Reference_2026.pdf` | 4 | Outlines non-covered services, permanent exclusions, conditional exclusions (such as adult dental/vision, bariatric surgery, and infertility), annual benefit limits, and Supplemental Shield pre-existing condition rules[cite: 2] |
| `MH_Terms_and_Conditions_2026.pdf` | 4 | Contains governing policy provisions covering general definitions, eligibility requirements, premium grace periods, waiting periods, guaranteed renewal, coverage termination, and COBRA continuation[cite: 3] |
| `MH_Claim_Process_Reimbursement_Guide_2026.pdf` | 5 | Details procedures for cashless and out-of-network reimbursement claims, documentation checklists, filing deadlines, out-of-network calculation steps, and two-level appeal processes[cite: 1] |
| `FAQ.pdf` | 4 | Provides internal representative reference answers for Plan Year 2026 customer queries covering cost-sharing copays, pre-existing condition rules, drug formulary tiers, claim filing processes, emergency care, prior authorization, and appeals[cite: 5] |

**Evaluation and Test Data Files**

| File | Queries | Description |
| -| -| -|
| `test.csv` | 20 | Test queries containing questions, expected answers, and relevant contexts used for pipeline evaluation |
| `evaluation.csv` | 10 | Benchmark evaluation queries with expected answers and relevant policy contexts used for final model validation |

# Installing and Importing the Necessary Libraries

In [ ]:
# Install all required packages using uv pip for fast dependency resolution
!uv pip install docling==2.126.0 pandas==2.2.3 transformers==5.16.1 langchain-huggingface==1.2.2 langchain-text-splitters==1.1.2 langchain-community==0.4.2 faiss-cpu==1.15.0 langchain-openai==1.6.1 deepeval==4.2.2 -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Import function for displaying outputs in Jupyter/Google Colab
from IPython.display import display

# Import library for working with tabular data
import pandas as pd

# Import Docling's document converter for processing PDF documents
from docling.document_converter import DocumentConverter

# Import tokenizer for determining token counts and preparing text for chunking
from transformers import AutoTokenizer

# Import LangChain's recursive character text splitter for token-aware chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import library for working with JSON data
import json

# Import LangChain's Document class for representing text and metadata
from langchain_core.documents import Document as LCDocument

# Import Hugging Face embedding model integration for creating vector embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Import OpenAI-compatible chat model for generating RAG responses
from langchain_openai import ChatOpenAI

# Import FAISS vector database for storing and retrieving document embeddings
from langchain_community.vectorstores import FAISS

# Import DeepEval evaluation framework
from deepeval import evaluate

# Import DeepEval test case class for defining evaluation inputs and outputs
from deepeval.test_case import LLMTestCase

# Import DeepEval metrics for evaluating different aspects of the RAG system

from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval

# Import OpenAI model wrapper used as the evaluation/judge model in DeepEval
from deepeval.models import OpenAIModel

# Import Python's operating system interface for managing environment variables
import os

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

/tmp/ipykernel_9832/2579019779.py:32: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# Data Understanding

## Policy Documents

**NOTE:** Before executing the code snippet below, create a folder named Data and upload all the dataset files (policy documents, FAQs, etc.) into it.

In [ ]:
# Define the root data path and list all policy document PDF file paths
root_path = 'Data'

pdf_paths = [
    root_path + "/" + "MH_Plan_Benefits_Coverage_Guide.pdf",
    root_path + "/" + "MH_Claim_Exclusion_Reference.pdf",
    root_path + "/" + "MH_Terms_and_Conditions.pdf",
    root_path + "/" + "MH_Claim_Process_Reimbursement_Guide.pdf"
]

### Plan,Benefits and Coverage Guide

- Features a title and metadata block detailing plan year, confidentiality notices, and regulatory state licensing.  

- Employs multi-column comparative tables to display deductibles, copays, and out-of-pocket maximums across four metal tiers (Bronze, Silver, Gold, Platinum).

- Uses structured topical headings to detail coverage definitions across all ten Essential Health Benefit (EHB) categories.

- Incorporates a dedicated tabular grid outlining prescription drug tiers, retail copays, and mail-order options.

- Includes separate standalone benefit tables for non-ACA plans (Supplemental Shield) alongside a companion document cross-reference section.

### Claim Exclusion Reference

- Opens with metadata reference codes, licensing scope, and a boxed policy warning notice regarding non-covered services.  

- Divides policy details into formal numbered divisions marked by explicit section tags (e.g., >> SECTION 1 through >> SECTION 6).  

- Utilizes bracketed alphanumeric tags (e.g., [EXC-2.1], [EXC-3.1]) to systematically index each specific exclusion.  

- Contains a structured Benefit Limitations table outlining visit caps, day limits, and plan applicability.  

- Employs bold text labels and callouts (e.g., EXCEPTION:) within text entries to specify coverage exceptions.

### Terms and Conditions

- Structured as a formal legal agreement with document metadata, versioning, and governing provision notices.  

- Segmented into ten distinct numbered sections (SECTION 1 through SECTION 10) governing contract terms.  

- Includes an alphabetized glossary in Section 1 defining key operational and legal terms.  

- Uses decimal sub-clause numbering (e.g., 2.1, 4.1, 7.1) to isolate provisions for individual plans, waiting periods, and appeals.  Integrates scenario-based text examples to illustrate complex timelines, such as waiting period calculations.

### Claim Process Reimbursement Guide

- Uses a standardized header displaying Document ID, version, effective date, state licensing, and corporate contact details.  

- Organizes workflows into step-by-step procedural guides (e.g., STEP 1 through STEP 5) for planned cashless care, out-of-network reimbursement, and Supplemental Shield claims.  

- Incorporates structured tables, including a Required Documentation Checklist matrix and a full Department Contact Directory.  

- Breaks down processes into level-based sections, such as Level 1 and Level 2 claim appeals.  Features step-by-step mathematical calculations accompanied by a concrete scenario example for out-of-network reimbursement.

## Evaluation and Test Data Files

### Evaluation Data

In [ ]:
# Load the evaluation dataset from CSV
file_path = 'Data/evaluation.csv'

df = pd.read_csv(file_path)
df

,question,expected_answer,relevant_context
0,What is the specialist visit copay for the Gold 80 plan?,"The specialist visit copay for the Gold 80 plan is $50. Per the plan's copayment rules, copayments for office visits (such as specialist visits) apply before the annual deductible is met, while other copayments apply after the deductible is met unless otherwise stated.","| Benefit | Bronze 60 | Silver 70 | Gold 80 | Platinum 90 |\n|---|---|---|---|---|\n| Annual Deductible (Individual) | $7,500 | $4,500 | $1,750 | $500 |\n| Annual Deductible (Family) | $15,000 | $9,000 | $3,500 | $1,000 |\n| Out-of-Pocket Max (Individual) | $9,450 | $8,200 | $6,800 | $3,500 |\n| Out-of-Pocket Max (Family) | $18,900 | $16,400 | $13,600 | $7,000 |\n| Primary Care Visit Copay | $45 | $35 | $25 | $15 |\n| Specialist Visit Copay | $80 | $65 | $50 | $30 |\n| Urgent Care Copay | $75 | $60 | $45 | $25 |\n| Emergency Room Copay | $450 | $350 | $250 | $150 |\n| Coinsurance (after deductible) | 40% | 30% | 20% | 10% |\n| Inpatient Hospital (per day) | $750/day | $500/day | $350/day | $200/day | <> Copayment (Copay). A fixed dollar amount the member pays at the time of receiving a covered service. Copayments apply before the deductible is met for office visits and prescription drugs; all other copayments apply after the deductible is met unless otherwise stated."
1,What does 'Medically Necessary' mean according to MeridianHealth's policy terms?,"Medically Necessary is defined as a service that is (a) required to diagnose or treat an illness, injury, or condition, (b) consistent with generally accepted medical practice, (c) not primarily for convenience, and (d) the most cost-effective level that can safely be provided. MeridianHealth's Medical Director makes the final determination on medical necessity.","Medically Necessary. A service that is (a) required to diagnose or treat an illness, injury, or condition, (b) consistent with generally accepted medical practice, (c) not primarily for convenience, and (d) the most cost-effective level that can safely be provided. MeridianHealth's Medical Director makes the final determination."
2,"Is a pre-existing condition like asthma covered from day one on an employer-sponsored plan, or is there a waiting period?","Once coverage is effective, asthma and all other pre-existing conditions are covered immediately with no condition-specific waiting period, since ACA-compliant plans do not impose waiting periods for pre-existing conditions and require no health questionnaire or medical underwriting. However, employer-sponsored group plans may impose a separate enrollment waiting period of up to 90 calendar days from the employee's date of hire (the employer selects 0, 30, 60, or 90 days) before coverage begins at all. This is an enrollment waiting period, not a condition-specific exclusion — once coverage starts, pre-existing condition benefits like asthma treatment are available with no further waiting.","4.1 ACA-Compliant Plans — No Condition-Specific Waiting Periods\nUnder the Affordable Care Act, MeridianHealth ACA-compliant plans do not impose any waiting period related to specific health conditions or pre-existing conditions. Once coverage is effective, all covered benefits are available immediately including benefits for pre-existing conditions such as diabetes, hypertension, cancer, heart disease, asthma, depression, and any other previously diagnosed condition. No health questionnaire, medical underwriting, or evidence of insurability is required for ACA plan enrollment. <> 4.2 Employer-Sponsored Plans — Enrollment Waiting Period\nEmployer-sponsored group plans may include an enrollment waiting period of up to 90 calendar days from the employee's date of hire, as permitted under ACA Section 2708. This is an enrollment waiting period, not a condition-specific exclusion. Once coverage becomes effective, all benefits including those for pre-existing conditions are available with no further waiting. The employer s

- The evaluation data contains the **question, expected answer, and relevant context** that the model should use to generate its response.

- In the relevant context, the chunks are separated by `<>` and are ordered by relevance, with the **most relevant chunk appearing first**.

- This evaluation set is required to **tune the parameters of our RAG pipeline** (if needed). Once we finalize the optimal parameters, we can perform the final evaluation on the test set.

In [ ]:
df['relevant_context'].apply(lambda x: len(x.split('<>'))).mean()

np.float64(2.2)

- The average number of relevant chunks is approximately **2**.
  - This provides a rough basis for choosing the number of chunks to be returned by the retrieval system, which will be defined later.

### Test Data

In [ ]:
# Load the test dataset from CSV
file_path = 'Data/test.csv'

df = pd.read_csv(file_path)
df

,question,expected_answer,relevant_context
0,What are the prescription drug copays for Tier 2 preferred brand medications at retail and mail order?,Tier 2 (Preferred Brand) prescription drugs cost a $35 copay for a 30-day retail fill and an $87.50 copay for a 90-day mail order supply.,| Tier | Category | Retail (30-day) | Mail Order (90-day) |\n|---|---|---|---|\n| Tier 1 | Generic | $10 copay | $25 copay |\n| Tier 2 | Preferred Brand | $35 copay | $87.50 copay |\n| Tier 3 | Non-Preferred Brand | $70 copay | $175 copay |\n| Tier 4 | Specialty | 30% coins. (max $300) | Not available by mail |
1,"How long does a member have to file an internal appeal after a claim denial, and how quickly will MeridianHealth respond?","The member must file an internal appeal within 180 calendar days of receiving the adverse benefit determination notice. MeridianHealth's decision timeframe is 30 calendar days for pre-service claims, 60 calendar days for post-service claims, and 72 hours for urgent care claims. The appeal is reviewed by a physician not involved in the original decision, and the member may submit additional documentation.",7.1 Internal Appeal\nThe member must file an internal appeal within 180 calendar days of receiving the adverse benefit determination notice. Reviewed by a physician not involved in the original decision. Standard decision: 30 calendar days (pre-service) or 60 calendar days (post-service). Urgent care: 72 hours. The member may submit additional documentation.
2,Are infertility treatments like IVF covered under the Silver plan?,"No. IVF and other infertility treatments (IUI, GIFT, ZIFT, donor procurement, and surrogacy costs) are excluded under both the Bronze and Silver plans. Only Gold and Platinum plans cover infertility treatment, up to 3 IVF cycles per lifetime with a $30,000 lifetime maximum, and prior authorization is required. Diagnostic infertility testing, however, is covered under all plan tiers. Note that state mandates may require additional coverage beyond what's described here.","[EXC-3.1]\nInfertility Treatment. IVF, IUI, GIFT, ZIFT, donor procurement, and surrogacy costs excluded under Bronze and Silver plans. Gold and Platinum plans cover up to 3 IVF cycles per lifetime with $30,000 lifetime max. Prior authorization required. Diagnostic testing covered under all tiers. State mandates may require additional coverage."
3,What is the annual out-of-pocket maximum for a family on the Gold 80 plan?,"The annual out-of-pocket maximum for a family on the Gold 80 plan is $13,600 (in-network). This is below the federal ACA ceiling of $21,200 for family coverage in Plan Year 2026.","| Benefit | Bronze 60 | Silver 70 | Gold 80 | Platinum 90 |\n|---|---|---|---|---|\n| Annual Deductible (Individual) | $7,500 | $4,500 | $1,750 | $500 |\n| Annual Deductible (Family) | $15,000 | $9,000 | $3,500 | $1,000 |\n| Out-of-Pocket Max (Individual) | $9,450 | $8,200 | $6,800 | $3,500 |\n| Out-of-Pocket Max (Family) | $18,900 | $16,400 | $13,600 | $7,000 |\n| Primary Care Visit Copay | $45 | $35 | $25 | $15 |\n| Specialist Visit Copay | $80 | $65 | $50 | $30 |\n| Urgent Care Copay | $75 | $60 | $45 | $25 |\n| Emergency Room Copay | $450 | $350 | $250 | $150 |\n| Coinsurance (after deductible) | 40% | 30% | 20% | 10% |\n| Inpatient Hospital (per day) | $750/day | $500/day | $350/day | $200/day | <> Note: All ACA-compliant plans observe the federal out-of-pocket maximum ceiling of $10,600 (individual) / $21,200 (family) for Plan Year 2026, per CMS guidance. The Supplemental Shield plan does not have an out-of-pocket maximum."
4,Can a member get cosmetic rhinoplasty covered under any MeridianHealth plan?,"No. Cosmetic rhinoplasty is permanently excluded under all MeridianHealth plans, since cosmetic surgery performed primarily to improve appearance (rather than restore function or treat a medical condition) is not covered. The exceptions are reconstructive surgery following a covered accident, injury, or mastectomy — covered und

The test set is used to **evaluate the final RAG pipeline** after the evaluation set has been used to assess the pipeline and tune its parameters, if required.

# Past Approach

In the past approach, MeridianHealth Insurance curated an FAQ based on the queries previously handled by the team. This serves as a good starting point, as repetitive queries can be handled effectively through simple lookups.

The FAQ is useful for **simple, repetitive queries**, but its coverage is limited for more complex real-world queries.

| Query Type | Example | FAQ Coverage | Limitation |
|---|---|---|---|
| **Direct** | “What is the deadline for submitting a reimbursement claim?” | FAQ-05 | Direct lookup works well. |
| **Direct** | “What are the copay amounts for a primary care visit under each plan tier?” | FAQ-01 | Direct lookup works well. |
| **Rephrased** | “My customer has diabetes and wants to know if they'll be covered right away on the Silver plan or if there's a waiting period.” | FAQ-02 | Requires semantic matching rather than exact lookup. |
| **Rephrased** | “A member went to a non-network ER after a car accident. Will they get hit with extra charges for being out of network?” | FAQ-06 | Requires understanding the query and mapping it to the relevant FAQ. |
| **Combination** | “A Gold plan member wants to get acupuncture for chronic back pain. Is it covered, does it need prior authorization, and how many sessions can they get?” | FAQ-12 + FAQ-08 (partial) + EXC-3.5 | FAQ provides only partial information; requires another policy document. |
| **Combination** | “A member on the Supplemental Shield plan was diagnosed with hypertension 8 months ago and is now hospitalized. What benefits can they claim and how do they file?” | FAQ-10 + FAQ-04/09 (partial) | Requires combining FAQ information with the Claim Process Guide. |
| **Outside FAQ** | “Can a 67-year-old retiree without any employer-sponsored coverage enroll in the Supplemental Shield plan?” | Not available | Requires Terms & Conditions, Section 2.3. |
| **Outside FAQ** | “How is the Maximum Allowable Amount calculated for out-of-network reimbursement, and what happens to charges that exceed it?” | Not available | Requires Terms & Conditions + Claim Process Guide. |

The FAQ handles **direct queries effectively**, but real-world queries often require **semantic matching, multi-document retrieval, information synthesis, or retrieval beyond the FAQ**. This highlights the need for a RAG-based approach.

# Chunking

The next step is to chunk these policy documents so we can pass them as context to the LLM. We cannot pass entire PDFs at once because doing so exceeds the context window limits.

Naive chunking by a fixed number of characters, regardless of where the cut falls, can split a table row, a numbered clause, or a heading right down the middle. A better default is to chunk **recursively**: try to split on larger, more natural boundaries first (blank lines, paragraph breaks), and only fall back to smaller units (sentences, words, characters) when a piece is still too large. This keeps most chunks aligned to natural text boundaries.

**Note:** Chunking as a concept is simple. We need to break documents into smaller pieces so they can be passed to the LLM within its context window. However, **how we split the documents depends on their structure**. Therefore, the right chunking strategy should be chosen based on the type and structure of the documents being used.


## Loading and Viewing Policy Documents with Docling

### Converting to Docling Document

- Docling gives us the PDF content in Markdown format, which preserves the structure, such as headings and tables.

- Other loaders, such as PyPDF and PyMuPDF, primarily extract the content as plain text.
    - If we send plain text to the LLM after chunking, the LLM may not be able to understand the table structure or the relationship between different sections.

The `DocumentConverter` is initialized with default settings. It will handle the conversion of each policy PDF into Docling's internal document representation, which preserves headings, tables, and structural elements and helps explore the PDF files in a structured manner

In [ ]:
# Initialize the Docling document converter
converter = DocumentConverter()

**Note:** The code snippet below may take approximately **8-10 minutes** to complete.

In [ ]:
# Convert each policy PDF into a Docling document object
docs = {}
for path in pdf_paths:
    result = converter.convert(path)
    docs[path] = result.document
    print(path," Completed ",end='\n')

[INFO] 2026-09-10 08:54:44,182 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-10 08:54:44,265 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-10 08:54:44,267 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-10 08:54:44,355 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-10 08:54:44,361 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-10 08:54:44,362 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-10 08:54:44,423 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-10 08:54:44,536 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Data/MH_Plan_Benefits_Coverage_Guide.pdf  Completed 
Data/MH_Claim_Exclusion_Reference.pdf  Completed 
Data/MH_Terms_and_Conditions.pdf  Completed 
Data/MH_Claim_Process_Reimbursement_Guide.pdf  Completed 


### Displaying the document as markdown

In [ ]:
# Display the last converted document as markdown to inspect its preserved structure
print(result.document.export_to_markdown())

## Claim Process &amp; Reimbursement Guide

Filing Claims | Required Documents | Timelines | Appeals

Plan Year 2026

DocumentID: MH-CPG-2026-047 | Effective: January1,2026| Version:3.1

Licensed in: CT, NY, NJ, PA, MA, MD, VA, NC, GA, FL, OH, IL, MI, TX, CO, AZ, WA, OR, CA, MN, WI, IN

## Overview of the Claim Process

MeridianHealth offers two pathways for claim settlement: cashless claims at in-network providers and reimbursement claims for out-of-network or non-participating provider services. For benefit amounts refer to the Plan Benefits Guide (MH-BRO-2026-047). For exclusions refer to the Claim Exclusion Reference (MH-EXC-2026-047). For definitions refer to the Terms and Conditions (MH-TC-2026-047, Section 1).

## Cashless Claims at In-Network Providers

When a member receives care at an in-network provider, MeridianHealth settles the covered portion directly with the provider. The member pays only the applicable copayment, coinsurance, or deductible amount.

## Planned (Non-Eme

* As seen in the output above, the PDF is represented as **Markdown**, with headings preserved using `##`, while tables are also retained in a structured format.
- This gives the text natural split points,blank lines between sections, headings on their own lines, and table rows rendered as Markdown,which the chunking step below can use to keep related content together.

### Displaying the tables

In [ ]:
# Extract and display all tables from the converted documents
for path, doc in docs.items():
    for item, _ in doc.iterate_items():
        if getattr(item, "label", "") == "table":
            print(f"\n--- Table from: {path} ---")
            print(item.export_to_markdown(doc=doc))


--- Table from: Data/MH_Plan_Benefits_Coverage_Guide.pdf ---
| Benefit                        | Bronze 60   | Silver 70   | Gold 80   | Platinum 90   |
|--------------------------------|-------------|-------------|-----------|---------------|
| Annual Deductible (Individual) | $7,500      | $4,500      | $1,750    | $500          |
| Annual Deductible (Family)     | $15,000     | $9,000      | $3,500    | $1,000        |
| Out-of-Pocket Max (Individual) | $9,450      | $8,200      | $6,800    | $3,500        |
| Out-of-Pocket Max (Family)     | $18,900     | $16,400     | $13,600   | $7,000        |
| Primary Care Visit Copay       | $45         | $35         | $25       | $15           |
| Specialist Visit Copay         | $80         | $65         | $50       | $30           |
| Urgent Care Copay              | $75         | $60         | $45       | $25           |
| Emergency Room Copay           | $450        | $350        | $250      | $150          |
| Coinsurance (after deducti

- As expected, the tables are retrieved with their **structure preserved**.
- When tables are included in a chunk, **Docling keeps the table structure intact rather than arbitrarily splitting it**, ensuring that the complete context and column relationships are preserved.


A chunk has two important parameters:

- **Chunk size:** The number of tokens allowed in a chunk.
- **Chunk overlap:** The number of tokens shared between consecutive chunks.

Ultimately, these chunks are passed to the **embedding model** and converted into vectors.

An important consideration is that **tokenization depends on the tokenizer**. For example, Tokenizer A may split text primarily based on spaces, while Tokenizer B may split differently based on punctuation or subwords. Therefore, the same sentence can result in a different number of tokens with different tokenizers.

Hence, we should use the **same tokenizer as the embedding model** when defining the chunk size, so that the token count used during chunking matches the embedding model's definition of a token.

## Choosing the embedding model

The next step is to choose an **embedding model** for our use case.

The model popular open-source models used for embedding text can be found [here](https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=trending), and majority of them can be utilized via the `sentence-transformers` library.

But do we always need to choose the most popular model? Not necessarily!

  - Embedding models are trained on different types and domains of text. **The quality of embeddings depends partly on how well the model's training data aligns with our domain.**

  - Since our documents are related to **medical insurance**, a domain-specific embedding model can potentially provide better representations than a general-purpose model.

  - For our use case, we can consider **`surajvbangera/mediclaim_embedding`**, available on Hugging Face, which is trained on **medical insurance documents**.

  - Therefore, instead of choosing an embedding model purely based on popularity, we should consider **how well it aligns with our data and retrieval task**.

In [ ]:
# define the model name
EMBED_MODEL = "surajvbangera/mediclaim_embedding"

# Load the domain-specific embedding model tokenizer and check its maximum token limit
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

print(f"Model max tokens: {tokenizer.model_max_length}")

Model max tokens: 512


From the above, it is clear that the **maximum number of tokens the embedding model can accept is 512**, based on its tokenizer.

Therefore, while chunking our documents, we can use the **same tokenizer used by the embedding model** and set the **maximum chunk size to 512 tokens**.

This ensures that every chunk generated is within the embedding model's supported token limit.

In [ ]:
# test how a sentence tokenizes
sample = "MeridianHealth offers four ACA-compliant metal tiers."
tokens = tokenizer.tokenize(sample)
print(f"Sample: '{sample}'")
print(f"Tokens ({len(tokens)}): {tokens}")

Sample: 'MeridianHealth offers four ACA-compliant metal tiers.'
Tokens (13): ['meridian', '##hea', '##lth', 'offers', 'four', 'ac', '##a', '-', 'compliant', 'metal', 'tier', '##s', '.']


- The sample sentence tokenizes into a small number of tokens, confirming that the tokenizer is working as expected.

- This also gives us a rough sense of how many tokens a typical policy sentence occupies within the 512 token chunk limit.

## Define the RecursiveCharacterTextSplitter

**LangChain's `RecursiveCharacterTextSplitter`** is well suited for these policy documents because it combines simplicity with sensible, structure-respecting defaults.

- **Recursive splitting:** It tries a prioritized list of separators (`\n\n`, `\n`, `". "`, `" "`, `""`) in order. It splits on the first separator that produces chunks within the size limit, so it prefers paragraph and sentence boundaries over arbitrary character cuts.

- **Works directly on text:** It operates on the plain Markdown text extracted by Docling from each PDF. Headings, paragraph breaks, and table rows appear as natural split points, so no document-specific chunk schema is required.

- **Token-aware:** Using `from_huggingface_tokenizer`, chunk size and overlap are measured in tokens from the same tokenizer used by the embedding model. This helps keep chunks within the embedding model's token limit.

- **Simple to tune:** Only two main parameters, `chunk_size` and `chunk_overlap`, need to be configured, making the splitter easy to experiment with and adjust.

The `RecursiveCharacterTextSplitter` is configured with:

- **512-token maximum (`chunk_size`)**: Matches the token limit of the embedding model. Using `from_huggingface_tokenizer`, chunk size is measured with the same tokenizer the embedding model uses, so it counts tokens the same way the embedding model will.

- **50-token overlap (`chunk_overlap`)**: Chosen as a starting value to preserve context between consecutive chunks. This can be tuned later based on the RAG pipeline's evaluation results.

- **Section-aware separators**: Since we observed earlier that the extracted Markdown preserves section headings using `##`, we can include `##` in the separator hierarchy. This allows the splitter to use these section boundaries when creating chunks, while still falling back to paragraph and sentence boundaries when needed.

This keeps chunks within the embedding model's token budget while respecting the document's section structure wherever possible.

In [ ]:
# Initialize the RecursiveCharacterTextSplitter using the same tokenizer as the embedding model
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,                  # Tokenizer used by the embedding model
    chunk_size=512,                      # Maximum number of tokens per chunk
    chunk_overlap=50,                    # Number of overlapping tokens
    separators=["\n\n", "\n", "##", ". ", " ", ""]
)

In [ ]:
# Wrap each document's Markdown export as a single LangChain Document, then
# split it into chunks using the RecursiveCharacterTextSplitter
raw_docs = [
    LCDocument(page_content=doc.export_to_markdown(), metadata={"source": path})
    for path, doc in docs.items()
]

lc_docs = splitter.split_documents(raw_docs)

print(f"Total chunks: {len(lc_docs)}")

Total chunks: 22


- All four policy documents have been chunked.

- The total chunk count reflects the combined chunks across all documents. Each chunk stays within the 512 token limit, measured using the embedding model's tokenizer, with 50 tokens of overlap between consecutive chunks.

In [ ]:
# Preview the text content of each generated chunk
for i, chunk in enumerate(lc_docs):
    print(f"\n--- Chunk {i} | {chunk.metadata['source']} ---")
    print(chunk.page_content[:400])
    print("..." if len(chunk.page_content) > 1000 else "")


--- Chunk 0 | Data/MH_Plan_Benefits_Coverage_Guide.pdf ---
## Plan Benefits &amp; Coverage Guide

Individual Family Employer-Sponsored Supplemental Plan Year 2026

DocumentID: MH-BRO-2026-047 Effective: January 1, 2026 Version 3.1

Licensed in: CT, NY, NJ, PA, MA, MD, VA, NC, GA, FL, OH, IL, MI, TX, CO, AZ, WA, OR, CA, MN, WI, IN

CONFIDENTIAL — For licensed representative use and policyholder reference only.

MeridianHealth Insurance Company | One Consti
...

--- Chunk 1 | Data/MH_Plan_Benefits_Coverage_Guide.pdf ---
| Benefit                        | Bronze 60   | Silver 70   | Gold 80   | Platinum 90   |
|--------------------------------|-------------|-------------|-----------|---------------|
| Annual Deductible (Individual) | $7,500      | $4,500      | $1,750    | $500          |
| Annual Deductible (Family)     | $15,000     | $9,000      | $3,500    | $1,000        |
| Out-of-Pocket Max (Individual) | $
...

--- Chunk 2 | Data/MH_Plan_Benefits_Coverage_Guide.pdf ---
Emergency 

- As seen in the previous output, each chunk retains the section heading along with its relevant content, while tables are also preserved in their structure.

- This provides the LLM with additional structural context when generating the response.

In [ ]:
# Preview the metadata of each generated chunk
for i, chunk in enumerate(lc_docs):
    print(f"\n--- Chunk {i} ---")
    print(chunk.metadata)


--- Chunk 0 ---
{'source': 'Data/MH_Plan_Benefits_Coverage_Guide.pdf'}

--- Chunk 1 ---
{'source': 'Data/MH_Plan_Benefits_Coverage_Guide.pdf'}

--- Chunk 2 ---
{'source': 'Data/MH_Plan_Benefits_Coverage_Guide.pdf'}

--- Chunk 3 ---
{'source': 'Data/MH_Plan_Benefits_Coverage_Guide.pdf'}

--- Chunk 4 ---
{'source': 'Data/MH_Plan_Benefits_Coverage_Guide.pdf'}

--- Chunk 5 ---
{'source': 'Data/MH_Plan_Benefits_Coverage_Guide.pdf'}

--- Chunk 6 ---
{'source': 'Data/MH_Claim_Exclusion_Reference.pdf'}

--- Chunk 7 ---
{'source': 'Data/MH_Claim_Exclusion_Reference.pdf'}

--- Chunk 8 ---
{'source': 'Data/MH_Claim_Exclusion_Reference.pdf'}

--- Chunk 9 ---
{'source': 'Data/MH_Claim_Exclusion_Reference.pdf'}

--- Chunk 10 ---
{'source': 'Data/MH_Claim_Exclusion_Reference.pdf'}

--- Chunk 11 ---
{'source': 'Data/MH_Terms_and_Conditions.pdf'}

--- Chunk 12 ---
{'source': 'Data/MH_Terms_and_Conditions.pdf'}

--- Chunk 13 ---
{'source': 'Data/MH_Terms_and_Conditions.pdf'}

--- Chunk 14 ---
{'source'

Each chunk carries a single metadata field:

- **`source`**: The path of the policy PDF the chunk came from, inherited from the parent document created before splitting.

This is enough to trace a retrieved chunk back to its source document when the LLM generates an answer. If page- or heading-level traceability is needed later, it can be added by splitting per page or by looking up page numbers from the original PDF as a post-processing step.

# Vector Database

Once the chunks are created, the next step is to **convert them into embeddings** and store them in a **Vector Database**. This allows the system to efficiently retrieve the most relevant chunks based on the user's query.

## Loading the embedding model

In [ ]:
# Load the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,  # Pre-trained embedding model
    model_kwargs={"device": "cpu"},                  # Run the model on CPU
    encode_kwargs={"normalize_embeddings": True},   # Normalize embeddings for similarity search
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Creating the Vector Database

Why **FAISS**?

- **Fast similarity search:** FAISS (Facebook AI Similarity Search) is built specifically for efficient nearest-neighbor search over dense vectors and scales well as the number of chunks grows.
- **Lightweight, in-memory index:** No separate database server or persistent collection to manage — the index lives in memory and can optionally be saved to and loaded from disk.
- **Good LangChain integration:** `langchain_community.vectorstores.FAISS` exposes the same `from_documents` / `similarity_search` interface as other vector stores, so the rest of the retrieval and evaluation code stays the same.
- **Widely used baseline:** FAISS is a common default choice for RAG prototypes and a solid baseline for comparison against managed or server-based vector databases.

**Note:** The code snippet below may take approximately **4-5 minutes** to complete.

In [ ]:
# Create a FAISS vector store from the LangChain documents
vectorstore = FAISS.from_documents(
    documents=lc_docs,
    embedding=embeddings,
)

# Save the index locally so it can be reloaded without rebuilding
vectorstore.save_local("faiss_index")

print(f"Vectors stored: {vectorstore.index.ntotal}")

Vectors stored: 22


## Retrieving top-k relevant chunks

In [ ]:
# Quick test

query = "What is the specialist visit copay for the Gold 80 plan?"


results = vectorstore.similarity_search(query, k=2)
for i, r in enumerate(results):
    print(f"\n--- Result {i} | {r.metadata.get('source')} ---")
    print(r.page_content)


--- Result 0 | Data/MH_Plan_Benefits_Coverage_Guide.pdf ---
| Benefit                        | Bronze 60   | Silver 70   | Gold 80   | Platinum 90   |
|--------------------------------|-------------|-------------|-----------|---------------|
| Annual Deductible (Individual) | $7,500      | $4,500      | $1,750    | $500          |
| Annual Deductible (Family)     | $15,000     | $9,000      | $3,500    | $1,000        |
| Out-of-Pocket Max (Individual) | $9,450      | $8,200      | $6,800    | $3,500        |
| Out-of-Pocket Max (Family)     | $18,900     | $16,400     | $13,600   | $7,000        |
| Primary Care Visit Copay       | $45         | $35         | $25       | $15           |
| Specialist Visit Copay         | $80         | $65         | $50       | $30           |
| Urgent Care Copay              | $75         | $60         | $45       | $25           |
| Emergency Room Copay           | $450        | $350        | $250      | $150          |
| Coinsurance (after deductib

- The similarity search returns the top-k chunks most relevant to the query.

- As shown above, the retrieved chunks are from the expected policy sections, confirming that the vector store and embedding model are working correctly for our domain.

# Retrieval System

In [ ]:
# Retrieve the top-k relevant chunks from the vector store
def retrieve(query: str, k: int = 2,vectorstore=None) -> list[str]:
    results = vectorstore.similarity_search(query, k=k)

    # Return each chunk with its content and metadata
    return [
        f"Content:\n{r.page_content}\n\nMetadata:\n{r.metadata}"
        for r in results
    ]

In [ ]:
# Test

query = "What is the specialist visit copay for the Gold 80 plan?"


for i, chunk in enumerate(retrieve(query,vectorstore=vectorstore)):
    print(f"\n--- Result {i} ---")
    print(chunk)


--- Result 0 ---
Content:
| Benefit                        | Bronze 60   | Silver 70   | Gold 80   | Platinum 90   |
|--------------------------------|-------------|-------------|-----------|---------------|
| Annual Deductible (Individual) | $7,500      | $4,500      | $1,750    | $500          |
| Annual Deductible (Family)     | $15,000     | $9,000      | $3,500    | $1,000        |
| Out-of-Pocket Max (Individual) | $9,450      | $8,200      | $6,800    | $3,500        |
| Out-of-Pocket Max (Family)     | $18,900     | $16,400     | $13,600   | $7,000        |
| Primary Care Visit Copay       | $45         | $35         | $25       | $15           |
| Specialist Visit Copay         | $80         | $65         | $50       | $30           |
| Urgent Care Copay              | $75         | $60         | $45       | $25           |
| Emergency Room Copay           | $450        | $350        | $250      | $150          |
| Coinsurance (after deductible) | 40%         | 30%         | 

> **Note:** The relevant chunks are returned along with their `source` metadata, i.e., the policy PDF they came from. This helps trace which document an answer's supporting evidence originated from when reviewing responses.

So far, we have defined **R, Retrieval**, which retrieves the relevant chunks from our policy documents.

Now, we need to define **G, Generator**, which uses the retrieved information as context to generate the final answer.

This is where **A, Augmentation**, comes in. We augment the LLM's prompt with the relevant retrieved context.

**RAG = Retrieval + Augmentation + Generation**

`User Query → Retrieve relevant chunks → Augment prompt with chunks → Generate answer`

# Generation System

## Loading and Testing the LLM

We load the OpenAI API credentials from `config.json` for use with the LLM and embedding model.

**Note**: Before executing the below code snippet, ensure that `config.json` is uploaded.

In [ ]:
# Load API configuration from the JSON config file
with open("config.json", "r") as f:
    config = json.load(f)

# Set environment variables for OpenAI API access
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]
os.environ["OPENAI_API_BASE"] = config["OPENAI_API_BASE"]

print("Configuration loaded.")

Configuration loaded.


In [ ]:
# Define the model configuration
MODEL = "gpt-4o-mini"
TEMPERATURE = 0
TOP_P = 1
MAX_TOKENS = 512

# Initialize the LLM with the defined configuration
llm = ChatOpenAI(
    model=MODEL,                              # Specify the LLM model
    temperature=TEMPERATURE,                  # Control randomness
    top_p=TOP_P,                              # Control the diversity of token selection
    max_tokens=MAX_TOKENS,                    # Set the maximum response length
    openai_api_key=config["OPENAI_API_KEY"],  # Provide the API key
    openai_api_base=config["OPENAI_API_BASE"] # Specify the API endpoint
)

In [ ]:
# Define a simple test query to verify LLM connectivity
TEST_QUERY_LLM = "Say 'LLM is ready' and nothing else."

# Send the test query to the LLM
response = llm.invoke(TEST_QUERY_LLM)

# Display the LLM response
print(response.content)

LLM is ready.


## Defining the generation pipeline

So far, we have defined the **retrieval step**, where relevant policy information is retrieved based on the user's query.

Next, we move to the **generation step**, where the LLM takes the **user query and the retrieved context** and generates the final response.

To guide the LLM towards producing a high-quality answer, we need to provide clear instructions on how it should use the retrieved information. We define these instructions as a **generation prompt** below.

> **Note:** For this step, we assume that the retrieved context is **relevant and of good quality**. Our focus here is on designing the generation prompt and guiding the LLM to produce the best possible response from the provided context.

In [ ]:
# Define the generation function that prompts the LLM with retrieved context
def generate(query, retrieved_chunks, model) -> str:

    prompt = f"""
    You are an AI assistant answering questions about medical insurance policies.

    User Query:
    {query}

    Retrieved Policy Information:
    {retrieved_chunks}

    Rules:
    1. Use only information explicitly supported by the retrieved policy information.
    2. Do not use outside knowledge, assumptions, or information not present in the retrieved policy information.
    3. Include all relevant information needed to answer the query completely.
    4. Do not omit important conditions, exclusions, limitations, exceptions, or coverage details.
    5. Focus only on information directly relevant to the user's query.
    6. Do not include unrelated information from the retrieved policy information.
    7. Answer the user's specific question directly and ensure the response addresses what was asked.
    8. Keep the answer concise, clear, and easy to understand.
    9. Do not invent benefits, exclusions, limits, waiting periods, conditions, or coverage details.
    10. If the information is insufficient, say:
        "The available policy documents do not contain enough information to answer this question."
        If the retrieved information conflicts, clearly mention the conflict.

    Answer:
    """

    return model.invoke(prompt)

Each instruction in the prompt relates to the key metrics specified earlier.

| Prompt Focus | Rules | Metric Addressed |
|---|---:|---|
| Use only supported information | 1-2 | Faithfulness |
| Include all relevant information | 3-4 | Contextual Recall |
| Focus on directly relevant information | 5-6 | Contextual Precision |
| Directly answer the user's question | 7 | Answer Relevancy |
| Keep the response concise and clear | 8 | Answer Relevancy |
| Avoid fabricated information | 9 | Faithfulness |
| Handle insufficient/conflicting information | 10 | Faithfulness + Contextual Recall |

# Retrieval Augmented Generation System

## Defining the RAG Function

In [ ]:
# Define the full RAG pipeline combining retrieval and generation
def rag(
    query: str,
    k: int = 2,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = 512,
    vectorstore=None
):
    # Create the generator model
    model = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens
    )

    # Retrieve relevant chunks
    retrieved_chunks = retrieve(query=query, k=k,vectorstore=vectorstore)

    # Generate answer using retrieved chunks
    answer = generate(
        query=query,
        retrieved_chunks=retrieved_chunks,
        model=model
    )

    return answer.content, retrieved_chunks

## Testing the RAG Pipeline

In [ ]:
# Test the RAG pipeline with a sample query

query = "What are the copay amounts for a primary care visit under each plan tier?"


response,rel_chunks = rag(
    query,vectorstore=vectorstore)

In [ ]:
# Print the generated response
print("Response: ",response)

Response:  The copay amounts for a primary care visit under each plan tier are as follows:

- **Bronze 60**: $45
- **Silver 70**: $35
- **Gold 80**: $25
- **Platinum 90**: $15


In [ ]:
# Display the retrieved chunks used to generate the response
rel_chunks

["Content:\nEmergency Services. Emergency room visits at any licensed facility — in-network or out-of-network — are covered at the in-network cost-sharing rate. If the patient is admitted directly from the ER, the ER copay is waived and inpatient hospital benefits apply.\n\nHospitalization. Inpatient hospital stays are covered subject to the per-day copay for the first five days, followed by coinsurance for each additional day. Semi-private room accommodation is the standard benefit. Private room upgrade is covered only when medically necessary.\n\nMaternity and Newborn Care. Prenatal visits, delivery (vaginal and cesarean), and postnatal care for mother and newborn are covered. Newborn coverage extends for the first 30 days under the mother's policy. The newborn must be enrolled as a dependent within 31 days of birth to continue coverage.\n\nMental Health and Substance Use Disorder Services. Outpatient therapy sessions are covered at the specialist copay rate. Inpatient behavioral hea

The RAG pipeline returns both the generated answer and the retrieved source chunks.

When needed, we can later use these source chunks to verify the answer.

# Evaluation Pipeline

## Loading the Evaluation Data

In [ ]:
# Load evaluation data
df = pd.read_csv("/content/Data/evaluation.csv")

## Initializing the Evaluation Model

In [ ]:
# Initialize the evaluation model using a stronger LLM for scoring

EVAL_MODEL = 'gpt-4o'

eval_model = OpenAIModel(
    model=EVAL_MODEL,
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_API_BASE"]
)

In [ ]:
# Verify the evaluation model is responding correctly
eval_model.generate('Hi')

('Hello! How can I assist you today?', 0.00011)

- The evaluation model is responding correctly.

- We use a stronger model (`gpt-4o`) for evaluation scoring, separate from the generation model (`gpt-4o-mini`), to ensure that the evaluation itself is reliable and not biased by the same model that generated the answers.

## Defining Evaluation Metrics

Four metrics are defined, each corresponding to one of the risks identified in the problem statement.
- **Faithfulness** guards against misrepresentation
- **Contextual Recall** checks retrieval completeness
- **Contextual Precision** validates that retrieved chunks are relevant and verifiable
- **Answer Relevancy** ensures the response addresses the actual question.

In [ ]:
# Faithfulness
# Checks whether the generated answer is factually supported by
# the information retrieved from the knowledge base.
faithfulness_metric = GEval(
    name="Faithfulness",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the actual output is factually supported by "
        "the provided retrieval context."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # First, identify the individual factual claims made in the answer.
        "Break the actual output into individual factual claims.",

        # Check each claim against the retrieved context.
        "Check whether each claim is supported by the retrieval context.",

        # Lower the score if the answer contains unsupported or conflicting claims.
        "Penalize claims that are unsupported or contradicted by the retrieval context.",

        # Differences in wording should not affect the score if the meaning is supported.
        "Do not penalize differences in wording when the meaning is supported by the context.",

        # Give a high score when the important claims are supported.
        "Give a higher score when all important claims are supported by the context."
    ],

    # Specifies which parts of the test case are used for evaluation.
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.85,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Contextual Recall
# Checks whether the retrieved context contains the information
# needed to produce the expected answer.
recall_metric = GEval(
    name="Contextual Recall",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the retrieval context contains the information "
        "needed to support the expected output."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Identify the important claims that the expected answer should contain.
        "Break the expected output into its important factual claims.",

        # Check whether those claims can be found or supported by the retrieved context.
        "Check whether each important claim is present or supported by the retrieval context.",

        # Lower the score when important information is missing from retrieval.
        "Penalize the retrieval context when important information required for the expected output is missing.",

        # Wording differences should not affect the evaluation.
        "Do not penalize differences in wording or presentation.",

        # Give a high score when retrieval contains most or all required information.
        "Give a higher score when the retrieval context contains most or all information needed to produce the expected output."
    ],

    # Recall requires the expected answer and retrieved context.
    evaluation_params=[
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.80,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Contextual Precision
# Checks whether the retrieved context is relevant to the user's question
# and whether relevant information is prioritized over irrelevant information.
precision_metric = GEval(
    name="Contextual Precision",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the retrieved context is relevant to answering "
        "the input question, with relevant information prioritized over irrelevant information."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Identify which parts of the retrieved context are useful for the question.
        "Identify the information in the retrieval context that is relevant to the input question.",

        # Check whether unnecessary or unrelated information was retrieved.
        "Check whether the retrieval context contains unnecessary or unrelated information.",

        # Lower the score when a large portion of the retrieved context is irrelevant.
        "Penalize retrievals that contain a large amount of irrelevant information.",

        # Give more credit when useful information appears prominently.
        "Give more credit when relevant information is prioritized.",

        # Give a high score when most retrieved information is useful.
        "Give a higher score when most of the retrieved context is useful for answering the question."
    ],

    # Precision evaluates the question against the retrieved context.
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.75,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Answer Relevancy
# Checks whether the generated answer directly addresses the user's question
# without including unnecessary or unrelated information.
answer_relevancy_metric = GEval(
    name="Answer Relevancy",

    # Defines what the evaluator should judge.
    criteria=(
        "Determine whether the actual output directly addresses the input "
        "question and provides an appropriate response without unnecessary information."
    ),

    # Step-by-step instructions given to the evaluator.
    evaluation_steps=[
        # Check whether the answer directly responds to the question.
        "Check whether the actual output directly answers the input question.",

        # Check whether the information provided is relevant to the question.
        "Check whether the response contains information relevant to the question.",

        # Lower the score for unrelated or unnecessary information.
        "Penalize unnecessary, unrelated, or off-topic information.",

        # A short answer should not be penalized if it answers the question sufficiently.
        "Do not penalize concise answers when they adequately address the question.",

        # Give a high score when the answer is focused and directly relevant.
        "Give a higher score when the response is focused and directly relevant to the question."
    ],

    # Answer relevancy evaluates the question against the generated answer.
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],

    # Minimum score required for the test case to pass.
    threshold=0.75,

    # LLM used to perform the evaluation.
    model=eval_model
)


# Store all evaluation metrics in a single list.
# This list can then be passed to DeepEval's evaluation function.
metrics = [
    faithfulness_metric,
    recall_metric,
    precision_metric,
    answer_relevancy_metric
]

## Building Evaluation Test Cases

In [ ]:
def test_case_generator(df, vectorstore, k=3):

    # Store the generated DeepEval test cases
    test_cases = []

    # Process each evaluation question
    for _, row in df.iterrows():

        # Run the RAG pipeline
        answer, retrieved_chunks = rag(
            query=row["question"],
            k=k,
            model_name=MODEL,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            vectorstore=vectorstore
        )

        # Split the ground-truth relevant context
        # Multiple contexts are separated by "<>"
        relevant_context = [
            c.strip()
            for c in row["relevant_context"].split("<>")
            if c.strip()
        ]

        # Create the DeepEval test case
        test_case = LLMTestCase(
            # User question
            input=row["question"],

            # Answer generated by the RAG pipeline
            actual_output=answer,

            # Ground-truth expected answer
            expected_output=row["expected_answer"],

            # Context actually retrieved by the RAG pipeline
            retrieval_context=retrieved_chunks,

            # Ground-truth relevant context
            context=relevant_context
        )

        # Add test case
        test_cases.append(test_case)

    return test_cases

## Running the Evaluation

In [ ]:
# Create the test cases

test_cases = test_case_generator(df,vectorstore)

> **Note:** The code below may occasionally throw an `AuthenticationError` related to the OpenAI API. If this error occurs, please re-run the code 2 to 4 times.

In [ ]:
# Run evaluation
results = evaluate(
    test_cases=test_cases,
    metrics=metrics
)

✨ You're running DeepEval's latest Faithfulness [GEval] Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is the specialist visit copay for the Gold 80 plan?                             │
│  │     Actual Output:      The specialist visit copay for the Gold 80 plan is $50.                              │
│  │     Expected Output:    The specialist visit copay for the Gold 80 plan is $50. Per the plan's copayment     │
│  │                         rules, copayments for office visits (such as specialist visits) apply before the     │
│  │                         annual deductible is met, while other copayments apply after the deductible is       │
│  │                         met unless otherwise stated.                                                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                       ┃ Score ┃ Threshold ┃ Reason                                        │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness [GEval]         │ 1.00  │ 0.85      │ The claim that the specialist visit copay     │
│              │                              │       │           │ for t...                                      │
│        PASS  │ Contextual Recall [GEval]    │ 0.86  │ 0.80      │ The retrieval context provides the            │
│              │                              │       │           │ specialist v...                               │
│        FAIL  │ Contextual Precision [GEval] │ 0.71  │ 0.75      │ The retrieval context contains the relevant   │
│              │                              │       │           │ information about the specialist visit        │
│              │                              │       │           │ copay for the Gold 80 plan, which is $50.     │
│              │                              │       │           │ However, it also includes a significant       │
│              │                              │       │           │ amount of unrelated information, such as      │
│              │                              │       │           │ details about other plans, benefit            │
│              │                              │       │           │ limitations, and exclusions, which are not    │
│              │                              │       │           │ necessary for answering the specific          │
│              │                              │       │           │ question. While the relevant information is   │
│              │                              │       │           │ present, the presence of extraneous details   │
│              │                              │       │           │ slightly detracts from the overall            │
│              │                              │       │           │ usefulness of the context.                    │
│        PASS  │ Answer Relevancy [GEval]     │ 1.00  │ 0.75      │ The response directly answers the input       │
│              │                              │       │           │ questio...                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=5985;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.29s | token cost: 0.184245 USD)
» Test Results (10 total tests):
   » Pass Rate: 90.0% | Passed: 9 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

**Note:** The evaluation results above are primarily obtained using an LLM-as-a-Judge approach. Since LLM-based evaluation can be non-deterministic, the metric scores and observations may vary slightly across different evaluation runs.

Therefore, the results from a single run should not be treated as definitive. Running the evaluation multiple times and looking at the overall trends can provide a more reliable estimate of the system's performance.

So far, we have evaluated the generated responses and obtained individual scores for **Faithfulness, Contextual Recall, Contextual Precision, and Answer Relevancy**.

To make a routing decision, we first combine these individual scores into a **single composite score** based on the predefined success criteria and weights.

The function below performs the following steps:

1. **Collects the evaluation scores** for each test case.
2. **Calculates the composite score** using the predefined weights:
   - Faithfulness: **35%**
   - Contextual Recall: **25%**
   - Contextual Precision: **20%**
   - Answer Relevancy: **20%**
3. **Compares the composite score against predefined thresholds**.
4. **Assigns a routing decision**:
   - **> 0.75 → AUTO ANSWER**
   - **0.50–0.75 → ASSISTED REVIEW**
   - **< 0.50 → MANUAL LOOKUP**
5. **Stores the question, individual metric scores, composite score, and routing decision** in a DataFrame for further analysis.

This allows us to move from individual evaluation metrics to an **overall decision on how the system should handle each query**.

In [ ]:
def create_evaluation_df(test_cases, results):

    rows = []

    # Loop through each test case and its corresponding evaluation result
    for test_case, result in zip(test_cases, results.test_results):

        # Create a dictionary of metric name -> score
        # Remove "[GEval]" from the metric names so they match our keys
        scores = {
            metric.name.replace(" [GEval]", "").strip().lower(): metric.score
            for metric in result.metrics_data
        }

        # Calculate the weighted composite score
        composite = (
            0.35 * scores["faithfulness"]
            + 0.25 * scores["contextual recall"]
            + 0.20 * scores["contextual precision"]
            + 0.20 * scores["answer relevancy"]
        )

        # Determine the routing type based on the composite score
        if composite > 0.75:
            routing_type = "AUTO ANSWER"
        elif composite >= 0.50:
            routing_type = "ASSISTED REVIEW"
        else:
            routing_type = "MANUAL LOOKUP"

        # Store the evaluation results for this question
        rows.append({
            "Question": test_case.input,
            "Faithfulness Score": scores["faithfulness"],
            "Contextual Recall Score": scores["contextual recall"],
            "Contextual Precision Score": scores["contextual precision"],
            "Answer Relevancy Score": scores["answer relevancy"],
            "Composite Score": composite,
            "Type": routing_type
        })

    return pd.DataFrame(rows)

In [ ]:
create_evaluation_df(test_cases, results)

,Question,Faithfulness Score,Contextual Recall Score,Contextual Precision Score,Answer Relevancy Score,Composite Score,Type
0,What is the specialist visit copay for the Gold 80 plan?,1.000000,0.864245,0.710564,1.000000,0.908174,AUTO ANSWER
1,What does 'Medically Necessary' mean according to MeridianHealth's policy terms?,1.000000,1.000000,0.803083,1.000000,0.960617,AUTO ANSWER
2,"Is a pre-existing condition like asthma covered from day one on an employer-sponsored plan, or is there a waiting period?",1.000000,0.973106,0.888346,0.783493,0.927644,AUTO ANSWER
3,What steps does a member need to follow to file a reimbursement claim for out-of-network services?,1.000000,1.000000,0.888246,0.835076,0.944664,AUTO ANSWER
4,Is bariatric surgery covered under the Bronze plan?,1.000000,0.919397,0.906178,0.983114,0.957708,AUTO ANSWER
5,"What is the waiting period for illness-related claims under the Supplemental Shield plan, and which benefits does it affect?",1.000000,0.892839,0.854674,0.939607,0.932066,AUTO ANSWER
6,What is the difference in annual deductible between the Bronze and Platinum plans for an individual member?,1.000000,1.000000,0.898489,0.993812,0.978460,AUTO ANSWER
7,"Does acupuncture require prior authorization on the Gold plan, and how many sessions are covered?",0.997702,0.892097,0.854301,0.963703,0.935821,AUTO ANSWER
8,"What is the maximum number of combined physical therapy, occupational therapy, and speech therapy visits covered per plan year?",0.996932,0.968595,0.886532,1.000000,0.968381,AUTO ANSWER
9,"If a member is admitted to an out-of-network hospital through the emergency room, how is the claim handled and what does the member owe?",1.000000,0.998831,0.862422,1.000000,0.972192,AUTO ANSWER


1. All 10 queries are classified as **AUTO ANSWER** (composite > 0.75). Faithfulness (≈1.00), Contextual Recall (≈0.95), and Answer Relevancy (≈0.95) are strong.

2. **Contextual Precision is the weakest metric (≈0.86)**. The specialist visit copay query scores 0.71, below the 0.75 threshold. The "Medically Necessary" definition query is also relatively low at 0.80.

3. Answer Relevancy dips slightly on compound questions, such as pre existing condition / waiting period (0.78) and out of network reimbursement steps (0.84), indicating partial coverage of multi part questions.

4. **9 of 10 queries pass all four metric thresholds**, with an overall composite average of ≈0.95, indicating strong performance before evaluation on the held out test set.

Based on the below guidelines, the retrieval and generation pipeline can be further tuned (if needed), and the evaluation can be re-run iteratively to identify and implement improvements in overall system performance.

| **Metric**                  | **Parameters to Tune**                                      | **Why These Parameters**                                                                                                                                                                                                 |
|-----------------------------|-------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Faithfulness**            | **LLM model, temperature, top_p, max_tokens, embedding model, k** | LLM model affects how well the answer stays grounded in the retrieved context. Lower temperature and controlled `top_p` reduce hallucinations. `max_tokens` helps prevent incomplete answers. The embedding model and `k` determine whether the correct supporting information is retrieved in the first place. |
| **Retrieval Completeness**  | **Embedding model, k, chunk size, chunk overlap**           | A better embedding model improves semantic matching. Increasing `k` retrieves more potentially relevant information. An appropriate chunk size preserves sufficient context, while chunk overlap helps retain information that falls across chunk boundaries. |
| **Retrieval Precision**     | **Embedding model, k, chunk size, chunk overlap**           | A better embedding model improves the relevance of retrieved chunks. Lowering `k` can reduce irrelevant results. Smaller, well-structured chunks can make retrieval more focused, while excessive overlap can introduce redundant information. |
| **Answer Relevancy**        | **LLM model, temperature, top_p, max_tokens, k**            | A stronger LLM can produce more direct and relevant answers. Lower temperature and controlled `top_p` help keep responses focused. `max_tokens` controls response length and prevents truncation. `k` determines how much context is provided to the LLM, which can affect how focused the answer is. |

# Evaluating on the Test set

## Loading the Test Data

In [ ]:
# Load evaluation data
df = pd.read_csv("Data/test.csv")

## Running the Evaluation

> **Note:** The code below may occasionally throw an `AuthenticationError` related to the OpenAI API. If this error occurs, please re-run the code 2 to 4 times.

In [ ]:
# Run evaluation

test_cases = test_case_generator(df,vectorstore)

results = evaluate(
    test_cases=test_cases,
    metrics=metrics
)

✨ You're running DeepEval's latest Faithfulness [GEval] Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy [GEval] Metric! (using gpt-4o, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                  

⚠ WARNING: No hyperparameters logged.
» ]8;id=227117;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 13.72s | token cost: 0.36507250000000013 USD)
» Test Results (20 total tests):
   » Pass Rate: 85.0% | Passed: 17 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
create_evaluation_df(test_cases, results)

,Question,Faithfulness Score,Contextual Recall Score,Contextual Precision Score,Answer Relevancy Score,Composite Score,Type
0,What are the prescription drug copays for Tier 2 preferred brand medications at retail and mail order?,1.000000,0.997838,0.886389,0.993440,0.975425,AUTO ANSWER
1,"How long does a member have to file an internal appeal after a claim denial, and how quickly will MeridianHealth respond?",1.000000,1.000000,0.876720,1.000000,0.975344,AUTO ANSWER
2,Are infertility treatments like IVF covered under the Silver plan?,1.000000,1.000000,0.913496,1.000000,0.982699,AUTO ANSWER
3,What is the annual out-of-pocket maximum for a family on the Gold 80 plan?,1.000000,0.995728,0.827070,1.000000,0.964346,AUTO ANSWER
4,Can a member get cosmetic rhinoplasty covered under any MeridianHealth plan?,1.000000,0.992523,0.877579,0.998776,0.973402,AUTO ANSWER
5,What is the maximum enrollment waiting period for a new employee joining an employer-sponsored group plan?,1.000000,0.997935,0.877805,1.000000,0.975045,AUTO ANSWER
6,A Supplemental Shield member had cancer diagnosed 10 months before enrollment. Are cancer-related hospitalizations covered under the Supplemental Shield?,0.991964,0.900000,0.275813,0.998550,0.827060,AUTO ANSWER
7,What documents are required to file a Supplemental Shield claim?,1.000000,0.994738,0.889037,1.000000,0.976492,AUTO ANSWER
8,How is the out-of-network reimbursement amount calculated for a Silver 70 plan member?,1.000000,1.000000,0.823180,1.000000,0.964636,AUTO ANSWER
9,"Are hearing aids covered under the Silver plan, and what are the limits?",0.998409,0.982446,0.914622,0.997702,0.977520,AUTO ANSWER


1. **19 of 20 queries (95%) are classified as AUTO ANSWER**, with 1 routed to ASSISTED REVIEW. Faithfulness (≈0.99) and Answer Relevancy (≈0.99) remain consistently high.

2. The single **ASSISTED REVIEW** query has the weakest scores overall: Faithfulness (0.80), Contextual Recall (0.27), and Contextual Precision (0.36), indicating a likely **retrieval gap**.

3. **Contextual Precision is the weakest aggregate metric (≈0.80)**. The Supplemental Shield pre existing cancer query scores 0.28, while preventive care zero cost sharing scores 0.67.

4. **17 of 20 queries (85%) pass all four thresholds**. The 3 failures involve exception or exclusion clauses or multi item benefit lists, suggesting these cases may require improved retrieval.

# Business Insights and Recommendations

## Business Insights

1. **Strong auto answer performance:** 19 of 20 queries (95%) qualify for AUTO ANSWER, well above the 50% target. Faithfulness is strong at **0.99**, above the 0.85 threshold.

2. **Strong case level quality:** 17 of 20 queries (85%) pass all four metric thresholds, showing that the high auto answer rate is supported by consistent metric performance.

3. **Retrieval precision is the main weakness:** **Contextual Precision (0.80)** is the weakest aggregate metric, particularly for exception clauses and multi item benefit lists, such as the Supplemental Shield pre existing cancer query (0.28) and reconstructive surgery query (0.36).

4. **One retrieval failure drives the only non auto answer:** The reconstructive surgery after mastectomy query fails Faithfulness, Recall, and Precision, indicating that the required supporting clause was likely not retrieved. The small test set of 20 queries limits generalization.

## Recommendations

1. **Improve retrieval precision and completeness:** Focus on exception clauses, pre existing conditions, coordination of benefits, and multi item benefit lists. Consider higher `k` or query rewriting for specific benefit queries.

2. **Improve multi part question handling:** Update the generation prompt to answer each sub question explicitly, or split compound queries into separate retrieval queries.

3. **Track case level quality:** Continue monitoring the **95% AUTO ANSWER rate**, but use the **85% all metrics pass rate** as the primary improvement metric.

4. **Expand the evaluation set:** Add more state specific plans, exclusions, conditional scenarios, and multi document questions. Follow a recurring **evaluate → fix → retest** cycle.